In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, rand, floor, concat, length, trim, to_date

In [0]:
# this is only for basic cleaning
class BaseCleaning:
    try:
        def __init__(self,df):
            self.df = df
        def lower_trim(self):
            for fields in self.df.schema:
                if isinstance(fields.dataType, StringType):
                    self.df = self.df.withColumn(fields,lower(trim(col(fields))))
    except Exception as e:
        print("you have erroe in base cleaning",e)

In [0]:
class Coustomer_info(BaseCleaning):
    try:
        def Coustmer_changes(self):
            renames = {
                            "cst_id":"coustmer_id",
                            "cst_key":"coustmer_key",
                            "cst_firstname":"coustmer_firstname",
                            "cst_lastname":"coustmer_lastname",
                            "cst_marital_status":"marriage_status",
                            "cst_gndr":"coustmer_gender",
                            "cst_create_date":"coustmer_join_date"
                }
            for old,new in renames.items():
                self.df = self.df.withColumn(old,new)
            #filtering Null values
            self.df = self.df.filter(col("coustmer_id").isNotNull())
            self.df = self.df.dropDuplicates(["coustmer_id"])
            #normilaation
            self.df = self.df.withColumn(
                                            "marriage_status",
                                                when(col("marriage_status") == "s","single")
                                                .when(col("marriage_statsus") == "m","married")
                                                .otherwise("n/a")
            )
            self.df = self.df.withColumn( "coustmer_gender",
                                                when(col("coustmer_gender") == "m","male")
                                                .when(col("coustmer_gender") == "female")

            )
            self.df = self.df.withColumn("data_added",current_date())
            print("sending the data to silver.cleaned_coustmer")
            return self.df.write.mode("append").format("delta").saveAsTable("silver.cleaned_coustmer")
    except Exception as e:
        print("you have error in coustmer_change",e)